# 02 — Joinen over bronze → silver → gold

De drie zone-catalogs delen dezelfde Hive Metastore, dus Trino kan over
de zone-grens heen joinen. Dit is dezelfde patroon als de dbt-models in
`dbt/models/uc01..uc10/` — alleen interactief vanuit een notebook.

Wat OPA toestaat hangt af van je rol. Als de query 'denied' geeft, run
dan `SELECT current_user` (notebook 01) om te zien onder welke identity
je werkt.

In [ ]:
from odap_lab import trino

# Hoeveel request per maand in silver, vs. wat we in gold hebben gerapporteerd?
trino.query('''
WITH silver_counts AS (
  SELECT date_trunc('month', aanvraag_datum) AS mnd, count(*) AS request
  FROM silver.claims.application
  GROUP BY 1
),
gold_counts AS (
  SELECT mnd, totaal_request
  FROM gold.uc01_claims_funnel
)
SELECT s.mnd, s.request AS silver_count, g.totaal_request AS gold_count
FROM silver_counts s
LEFT JOIN gold_counts g ON g.mnd = s.mnd
ORDER BY s.mnd
LIMIT 24
''')

In [ ]:
# Resultaat doorgeven aan DuckDB voor in-process aggregatie / pivot.
import duckdb
from odap_lab import trino

df = trino.query('SELECT * FROM gold.uc01_claims_funnel LIMIT 10000')
duckdb.query("SELECT count(*), avg(totaal_request) FROM df").df()

In [ ]:
# Bronze rechtstreeks bevragen — alleen toegestaan met JIT of als platform-admin.
try:
    print(trino.query('SELECT count(*) FROM bronze.persons.created').iloc[0, 0])
except Exception as exc:
    print('Bronze geblokkeerd door OPA — verwacht for the meeste rollen:')
    print(repr(exc))